# 03 — Train Random Forest

Loads `features.parquet`, splits subjects into train + held-out test, runs 5-fold GroupKFold CV inside the train pool, and persists the best model and metrics.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

from bme_ml.paths import setup_paths
from bme_ml.splits import make_splits, select_rows, add_subject_id
from bme_ml.models import build_rf, tune_rf, Xy
from bme_ml.features import FEATURE_NAMES
from bme_ml.evaluation import evaluate_binary, feature_importance

paths = setup_paths()

In [ ]:
features = pd.read_parquet(paths.features_parquet)
features = add_subject_id(features)
print('rows:', len(features), 'subjects:', features['subject_id'].nunique())

splits = make_splits(features, label_col='label_binary')
splits.to_json(paths.splits_json)
print('test subjects:', len(splits.test_subjects), 'cv folds:', len(splits.cv_folds))

In [ ]:
train_subjects = sorted({s for fold in splits.cv_folds for s in fold['train']} |
                        {s for fold in splits.cv_folds for s in fold['val']})
train_df = select_rows(features, train_subjects)
test_df = select_rows(features, splits.test_subjects)

X_train, y_train = Xy(train_df)
X_test, y_test = Xy(test_df)
print('train:', X_train.shape, 'test:', X_test.shape)

In [ ]:
# Build CV indices manually so GridSearchCV honors subject-level folds.
cols = [c for c in FEATURE_NAMES if c in train_df.columns]
train_clean = train_df.dropna(subset=cols + ['label_binary']).reset_index(drop=True)
groups = train_clean['subject_id'].to_numpy()
gkf = GroupKFold(n_splits=5)
cv_splits = list(gkf.split(train_clean, groups=groups))

grid = tune_rf(X_train, y_train, cv_splits)
print('best params:', grid.best_params_)
print('best CV f1_macro:', grid.best_score_)

In [ ]:
model = grid.best_estimator_
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]
metrics = evaluate_binary(y_test, y_pred, y_proba)
print(metrics)

joblib.dump(model, paths.models / 'rf_binary.joblib')
(paths.processed / 'rf_metrics.json').write_text(json.dumps(metrics.__dict__, indent=2))
print('saved model + metrics')

In [ ]:
imp = feature_importance(model, X_test, y_test, feature_names=cols)
imp